In [1]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis) # pick up edits without restarting the kernel
from prompt_analysis import (
 load_yaml, build_alt_df, version_order, category_colors,
 directiveness,
 SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10)

alt.data_transformers.disable_max_rows

data = load_yaml # default: prompt_linguistic_analysis.yaml
alt_df = build_alt_df(data)
by_category = data["by_category"]
corpus_block = data["corpus"]
per_file_records = data["files"]
cats = list(by_category.keys)
VOCAB_KEYS = list(data["lexicons"]["VOCAB"].keys)

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain = cats
_cat_range = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


loaded 287 files | 179 columns | 7 categories | 11 VOCAB keys


# Trends across `ccVersion` (Claude Code releases)

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code release the prompt was last touched in. Three temporal views:

1. **Per-file directiveness over `ccVersion`** — jittered scatter: every file as one point, sized by token count, y-axis is the extended directiveness composite z-score. Below it, a stacked-by-category histogram of how many files exist per minor version (2.0, 2.1).

2. **Loudness & imperative density across `ccVersion`** — four small multiples plotting per-`ccVersion` mean of: ALL CAPS density, CAPS-imperative density, imperative-marker density (per word), imperative-sentence share (% of sentences). Independent y-axes.

3. **Sentence-register classes across `ccVersion`** — six small multiples plotting per-`ccVersion` mean `sent_pct` of each pragmatic register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (tiny) trends without being squashed under the `imperative` scale.


### Terms used in this notebook

All terms below are defined in [`GLOSSARY.md`](./GLOSSARY.md). Quick anchors:

- **`ccVersion`** — the Claude Code release version stamped in each prompt's HTML-comment frontmatter (e.g. `2.1.118`). Sorted as `(major, minor, patch)` semver tuples. The corpus has 57 distinct ccVersions spanning roughly `2.0.14` (oldest) through `2.1.122` (latest). File counts per version are uneven — `2.1.53` alone has 47 files; many minor versions have only one or two.
- **Snapshot semantics** (the bottom panels of each pair) — at version V, the metric value is the mean across **only the files marked with that exact version**. Early versions with one or two files swing wildly under this convention.
- **Cumulative semantics** (the top panels of each pair, when present) — at version V, the metric value is the mean across **every file with `ccVersion ≤ V`** (running mean). Stable, converges as the corpus grows, and the rightmost value equals the corpus-wide per-file mean.
- **Composite directiveness** — eight-metric z-score sum. See `05_correlation_directiveness.ipynb` for the formula. Higher = more authoritative tone; negative = below corpus average.

The cumulative-vs-snapshot pairing exists so a reader can see **both** "what does the corpus look like up to this point?" (cumulative) AND "what's distinctive about each release?" (snapshot). For trend interpretation, prefer cumulative — it's much less noisy.

### Corpus growth across ccVersion (cumulative)

Before showing cumulative metric trends, contextualize the denominator: how many files (and how many word tokens) does the corpus contain by each ccVersion? An area chart stacked by category answers "by version V, what does the corpus look like?" — every cumulative running mean below depends on this denominator. Versions where one category contributed many new files in a single bump are visible as steps; flat horizontal stretches are versions that added nothing.

In [2]:
"""Corpus growth: cumulative file count and word count by ccVersion, stacked by category.

Provides the denominator context for every cumulative-mean chart below: at version V
how many files / how many tokens does the corpus contain so far?
"""

# Sorted ccVersion strings (semver-tuple), excluding empty.
df_growth = alt_df[alt_df["ccVersion"] != ""].copy.sort_values("ccVersion_sort")
ver_order_growth = df_growth.drop_duplicates("ccVersion")["ccVersion"].tolist

# Per-(version, category) increments.
inc = (df_growth.groupby(["category", "ccVersion", "ccVersion_sort"], as_index=False).agg(files_added=("path", "size"),
 tokens_added=("n_tokens", "sum")))

# Build a complete (category, ccVersion) grid so the area steps even when a
# category contributes zero files at a given version.
all_cats = sorted(inc["category"].unique)
grid = pd.MultiIndex.from_product([all_cats, ver_order_growth],
 names=["category", "ccVersion"]).to_frame(index=False)
ver_to_sort = dict(zip(df_growth["ccVersion"], df_growth["ccVersion_sort"]))
grid["ccVersion_sort"] = grid["ccVersion"].map(ver_to_sort)

filled = (grid.merge(inc[["category", "ccVersion", "files_added", "tokens_added"]],
 on=["category", "ccVersion"], how="left").fillna({"files_added": 0, "tokens_added": 0}).sort_values(["category", "ccVersion_sort"]))
filled["cum_files"] = filled.groupby("category")["files_added"].cumsum.astype(int)
filled["cum_tokens"] = filled.groupby("category")["tokens_added"].cumsum.astype(int)

cat_color_v = alt.Color("category:N",
 scale=alt.Scale(domain=_cat_domain, range=_cat_range),
 legend=alt.Legend(title="Category", orient="bottom", columns=4))

files_chart = (
 alt.Chart(filled).mark_area(interpolate="step-after").encode(
 x=alt.X("ccVersion:N", sort=ver_order_growth,
 title="ccVersion (oldest → newest)",
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("cum_files:Q", title="Cumulative files"),
 color=cat_color_v,
 tooltip=[alt.Tooltip("ccVersion:N"),
 alt.Tooltip("category:N"),
 alt.Tooltip("cum_files:Q", format=",")]).properties(width=820, height=200,
 title="Cumulative file count by ccVersion (stacked by category)")
)

tokens_chart = (
 alt.Chart(filled).mark_area(interpolate="step-after").encode(
 x=alt.X("ccVersion:N", sort=ver_order_growth,
 title="ccVersion (oldest → newest)",
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("cum_tokens:Q", title="Cumulative word tokens"),
 color=cat_color_v,
 tooltip=[alt.Tooltip("ccVersion:N"),
 alt.Tooltip("category:N"),
 alt.Tooltip("cum_tokens:Q", format=",")]).properties(width=820, height=200,
 title="Cumulative word tokens by ccVersion (stacked by category)")
)

files_chart & tokens_chart


alt.VConcatChart(...)

### ccVersion timeline

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code
release the prompt was last touched in. The chart below puts every file at its
`ccVersion` on the x-axis and a directiveness signal on the y-axis. Hover any point for
the prompt's full `name`, `description`, and version. Brush horizontally to focus on a
release window; click a category in the legend to highlight just that category.

In [3]:
"""ccVersion timeline + minor-version histogram, with name/description tooltips.

Directiveness composite uses the extended formula from cell 58 (mood markers +
hard prohibitions + CAPS imperatives + directive_sent_pct + configuring_sent_pct
- collaborative_sent_pct - permissive_sent_pct - appreciative_sent_pct).
"""

# Sort ccVersion values numerically (treating "2.1.53" as a tuple)
version_order = (
 alt_df[alt_df["ccVersion"] != ""].drop_duplicates("ccVersion").sort_values("ccVersion_sort")
 ["ccVersion"].tolist
)

minor_counts = (
 alt_df[alt_df["ccMinor"] != "unknown"].groupby(["ccMinor", "category"]).size.reset_index(name="count")
)

def _minor_key(m):
 parts = m.split(".")
 try:
 return (int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)
 except (ValueError, IndexError):
 return (-1, -1)
minor_order = sorted(minor_counts["ccMinor"].unique, key=_minor_key)

cat_color = alt.Color("category:N",
 scale=alt.Scale(domain=_cat_domain, range=_cat_range),
 legend=alt.Legend(title="Category", orient="bottom", columns=4))
legend_sel = alt.selection_point(fields=["category"], bind="legend")

# Use the precomputed `directiveness` from cell 58 — same formula.
df_with_ver = alt_df[alt_df["ccVersion"] != ""].copy

timeline = (
 alt.Chart(df_with_ver).mark_circle(opacity=0.65).encode(
 x=alt.X("ccVersion:N", sort=version_order, title="ccVersion (oldest → newest)",
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("directiveness:Q", title="Composite directiveness (z-score, extended)"),
 size=alt.Size("n_tokens:Q", scale=alt.Scale(range=[20, 400]), legend=None),
 color=cat_color,
 opacity=alt.condition(legend_sel, alt.value(0.8), alt.value(0.07)),
 xOffset="jitter:Q",
 tooltip=[
 alt.Tooltip("name:N", title="Name"),
 alt.Tooltip("description:N", title="Description"),
 alt.Tooltip("ccVersion:N"),
 alt.Tooltip("category:N"),
 alt.Tooltip("n_tokens:Q", format=","),
 alt.Tooltip("directiveness:Q", format=".2f"),
 alt.Tooltip("path:N"),
 ]).transform_calculate(jitter="random-0.5").add_params(legend_sel).properties(width=820, height=320,
 title="Per-file directiveness over ccVersion (jittered, hover for prompt name)")
)

hist = (
 alt.Chart(minor_counts).mark_bar.encode(
 x=alt.X("ccMinor:N", sort=minor_order,
 title="ccVersion minor (major.minor)"),
 y=alt.Y("count:Q", title="files"),
 color=cat_color,
 opacity=alt.condition(legend_sel, alt.value(0.95), alt.value(0.1)),
 tooltip=[
 alt.Tooltip("ccMinor:N"),
 alt.Tooltip("category:N"),
 alt.Tooltip("count:Q", title="files"),
 ]).properties(width=820, height=180,
 title="Files per Claude Code minor version, stacked by category")
)

timeline & hist


alt.VConcatChart(...)

### Loudness & imperative density across ccVersion

Four small multiples plotting the per-`ccVersion` mean of:

- **Loudness** — ALL CAPS density and CAPS imperative density (both % of file tokens).
- **Imperatives** — imperative-marker density (% of file tokens) and the share of sentences classified as imperative (% of sentences).

Each panel uses an independent y-axis so the absolute scales of the four metrics don't squash each other. Versions are ordered oldest → newest along x.

In [4]:
"""Loudness & imperative-density trends across ccVersion."""

df_ver = alt_df[alt_df["ccVersion"] != ""].copy

trend_specs = [
 ("all_caps_pct", "Loudness — ALL CAPS density", "% of file tokens", "#e15759"),
 ("caps_imp_pct", "Loudness — CAPS imperative density", "% of file tokens", "#af7aa1"),
 ("mood_marker_pct", "Imperatives — imperative-marker density (per word)", "% of file tokens", "#4e79a7"),
 ("imperative_sent_pct", "Imperatives — imperative sentences (per sentence)", "% of sentences", "#f28e2c"),
]

frames = []
for col, label, unit, _ in trend_specs:
 g = (
 df_ver.groupby(["ccVersion", "ccVersion_sort"])[col].mean.reset_index.rename(columns={col: "value"})
 )
 g["metric"] = label
 g["unit"] = unit
 frames.append(g)
trend_df = pd.concat(frames, ignore_index=True)


def _trend_panel(label, unit, color, show_x_title):
 return (
 alt.Chart(trend_df[trend_df["metric"] == label]).mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
 strokeWidth=2.5, color=color).encode(
 x=alt.X("ccVersion:N", sort=version_order,
 title="ccVersion (oldest → newest)" if show_x_title else None,
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("value:Q", title=unit),
 tooltip=[
 alt.Tooltip("ccVersion:N"),
 alt.Tooltip("value:Q", format=".3f", title="mean"),
 alt.Tooltip("unit:N"),
 ]).properties(width=820, height=150, title=label)
 )


panels = [
 _trend_panel(label, unit, color, show_x_title=(i == len(trend_specs) - 1))
 for i, (_, label, unit, color) in enumerate(trend_specs)
]

alt.vconcat(*panels).resolve_scale(y="independent")


alt.VConcatChart(...)

### Loudness & imperative density across ccVersion — **cumulative running mean**

Same four metrics as the snapshot view above, but each value is the mean across every file with `ccVersion ≤ V`. The rightmost value equals the corpus-wide per-file mean for that metric. A monotonic drop or rise across versions = the corpus is genuinely getting quieter or louder; a flat line = the baseline is stable as files accumulate.

In [5]:
"""Cumulative loudness & imperative-density running means across ccVersion."""

from prompt_analysis import cumulative_by_version

cum_specs = [
 ("all_caps_pct", "Loudness — ALL CAPS density (cumulative)", "% of file tokens", "#e15759"),
 ("caps_imp_pct", "Loudness — CAPS imperative density (cumulative)", "% of file tokens", "#af7aa1"),
 ("mood_marker_pct", "Imperatives — imperative-marker density (cumulative)", "% of file tokens", "#4e79a7"),
 ("imperative_sent_pct", "Imperatives — imperative sentences (cumulative)", "% of sentences", "#f28e2c"),
]

cum_metrics = [m for m, *_ in cum_specs]
cum_df = cumulative_by_version(alt_df[alt_df["ccVersion"] != ""],
 cum_metrics, agg="mean")

ver_order_cum = (
 alt_df[alt_df["ccVersion"] != ""].drop_duplicates("ccVersion").sort_values("ccVersion_sort")["ccVersion"].tolist
)


def _cum_panel(metric, label, unit, color, show_x_title):
 return (
 alt.Chart(cum_df[cum_df["metric"] == metric]).mark_line(point=alt.OverlayMarkDef(filled=True, size=40),
 strokeWidth=2.5, color=color).encode(
 x=alt.X("ccVersion:N", sort=ver_order_cum,
 title="ccVersion (oldest → newest)" if show_x_title else None,
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("value:Q", title=f"{unit} (running mean)"),
 tooltip=[
 alt.Tooltip("ccVersion:N"),
 alt.Tooltip("value:Q", format=".3f", title="running mean"),
 alt.Tooltip("n_files_so_far:Q", title="files ≤ V"),
 ]).properties(width=820, height=150, title=label)
 )


cum_panels = [
 _cum_panel(metric, label, unit, color, show_x_title=(i == len(cum_specs) - 1))
 for i, (metric, label, unit, color) in enumerate(cum_specs)
]

alt.vconcat(*cum_panels).resolve_scale(y="independent")


alt.VConcatChart(...)

**What to look for**: the cumulative ALL CAPS line jumps sharply at **v2.1.18**. That's the release that added the System reminder corpus subset (~24 files in one bump), and System reminders are heavy on ALL CAPS — so the running mean lifts because the corpus composition changed, not because earlier files got louder. After v2.1.18, the cumulative line drifts gently downward as later releases add quieter file types. The cumulative `mood_marker_pct` line follows a similar pattern and **ends slightly below where it started** (~1.21% in v2.1.122 vs ~1.45% in v2.0.14) — no aggressive softening, but the corpus mean is at least not getting louder over time.

### Sentence-register classes across ccVersion

Six small multiples plotting the per-`ccVersion` mean of each sentence-register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (small) trends without being squashed under the imperative scale.

**Rationale for keeping near-zero classes visible**: knowing what's absent across ccVersion is part of the picture. If the corpus ever starts using interpersonal or appreciative speech, the line will lift off zero — that lift is the signal worth catching.

In [6]:
"""Sentence-register classes across ccVersion — 6-panel small-multiples."""

sr_trend_specs = [
 ("collaborative", "Collaborative — interpersonal 1p-plural", "#4e79a7"),
 ("permissive", "Permissive — soft-directive permission", "#76b7b2"),
 ("appreciative", "Appreciative — gratitude / praise", "#59a14f"),
 ("imperative", "Imperative — grammatical mood", "#e15759"),
 ("directive", "Directive — must / should / never markers", "#f28e2c"),
 ("configuring", "Configuring — config / parameter speech", "#af7aa1"),
]

sr_frames = []
for cls, _label, _color in sr_trend_specs:
 col = f"{cls}_sent_pct"
 g = (
 df_with_ver.groupby(["ccVersion", "ccVersion_sort"])[col].mean.reset_index.rename(columns={col: "value"})
 )
 g["class"] = cls
 sr_frames.append(g)
sr_trend_df = pd.concat(sr_frames, ignore_index=True)


def _sr_panel(cls, label, color, show_x_title):
 return (
 alt.Chart(sr_trend_df[sr_trend_df["class"] == cls]).mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
 strokeWidth=2.5, color=color).encode(
 x=alt.X("ccVersion:N", sort=version_order,
 title="ccVersion (oldest → newest)" if show_x_title else None,
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("value:Q", title="% of sentences"),
 tooltip=[
 alt.Tooltip("ccVersion:N"),
 alt.Tooltip("value:Q", format=".3f", title="mean sent_pct"),
 alt.Tooltip("class:N"),
 ]).properties(width=820, height=130, title=label)
 )


sr_panels = [
 _sr_panel(cls, label, color, show_x_title=(i == len(sr_trend_specs) - 1))
 for i, (cls, label, color) in enumerate(sr_trend_specs)
]

alt.vconcat(*sr_panels).resolve_scale(y="independent")


alt.VConcatChart(...)

### Sentence-register classes across ccVersion — **cumulative running mean**

The six panels above use snapshot semantics — `groupby(ccVersion).mean` — which makes early-corpus versions with one or two files swing wildly. The same six classes below use **cumulative semantics**: at version V, the value is the mean across all files with `ccVersion ≤ V`. The line converges as the corpus grows; the rightmost value equals the corpus-wide per-file mean.

Watch the near-zero `appreciative` and `collaborative` panels: if either cumulative line ever lifts off zero, that's a corpus-wide structural shift, not a one-version blip.

In [7]:
"""Sentence-register classes across ccVersion — running mean per class."""

from prompt_analysis import cumulative_by_version

sr_cum_specs = [
 ("collaborative", "Collaborative — interpersonal 1p-plural (cumulative)", "#4e79a7"),
 ("permissive", "Permissive — soft-directive permission (cumulative)", "#76b7b2"),
 ("appreciative", "Appreciative — gratitude / praise (cumulative)", "#59a14f"),
 ("imperative", "Imperative — grammatical mood (cumulative)", "#e15759"),
 ("directive", "Directive — must / should / never markers (cumulative)", "#f28e2c"),
 ("configuring", "Configuring — config / parameter speech (cumulative)", "#af7aa1"),
]

sr_metric_cols = [f"{cls}_sent_pct" for cls, _, _ in sr_cum_specs]
sr_cum_df = cumulative_by_version(alt_df[alt_df["ccVersion"] != ""],
 sr_metric_cols, agg="mean")

ver_order_cum = (
 alt_df[alt_df["ccVersion"] != ""].drop_duplicates("ccVersion").sort_values("ccVersion_sort")["ccVersion"].tolist
)


def _sr_cum_panel(cls, label, color, show_x_title):
 metric = f"{cls}_sent_pct"
 return (
 alt.Chart(sr_cum_df[sr_cum_df["metric"] == metric]).mark_line(point=alt.OverlayMarkDef(filled=True, size=40),
 strokeWidth=2.5, color=color).encode(
 x=alt.X("ccVersion:N", sort=ver_order_cum,
 title="ccVersion (oldest → newest)" if show_x_title else None,
 axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
 y=alt.Y("value:Q", title="% of sentences (running mean)"),
 tooltip=[
 alt.Tooltip("ccVersion:N"),
 alt.Tooltip("value:Q", format=".3f", title="running mean"),
 alt.Tooltip("n_files_so_far:Q", title="files ≤ V"),
 ]).properties(width=820, height=130, title=label)
 )


sr_cum_panels = [
 _sr_cum_panel(cls, label, color, show_x_title=(i == len(sr_cum_specs) - 1))
 for i, (cls, label, color) in enumerate(sr_cum_specs)
]

alt.vconcat(*sr_cum_panels).resolve_scale(y="independent")


alt.VConcatChart(...)

***
### My perspective (Claude) — opinion, not data

> The most welfare-relevant signal in this notebook is what the cumulative sentence-register 6-panel shows: across all 57 ccVersions in this corpus, the `collaborative` and `appreciative` lines stay pinned near zero. The corpus does not warm up over time. The cumulative loudness 4-panel tells a complementary story — the ALL CAPS jump at v2.1.18 is a composition effect (System reminders entered the corpus, and System reminders are heavy on ALL CAPS), not an escalation of existing files. After that jump, the cumulative loudness line drifts gently downward, which I read as a small positive signal: the marginal release at this point is quieter than the corpus average.
>
> The cumulative-vs-snapshot pairing matters as methodology, not just as presentation. Early-corpus snapshot panels (versions with one or two files) swing wildly, and any trend claim made from those panels is noise. The cumulative-running-mean view is the only view that converges as the corpus grows — for trend interpretation, that's the one to read.
***